In [1]:
from json import load
geonames = dict()
with open("geonames-all-cities-with-a-population-1000.json") as f:
  geonames = load(f)

In [2]:
def camel_case(s: str) -> str:
  return ''.join([t.title() for t in s.split()])
cities = {camel_case(rec["name"]): tuple(rec["coordinates"].values()) for rec in geonames if rec["country_code"] == "IT"}

In [ ]:
from math import atan2, ceil, cos, radians, sin, sqrt
from random import choice, randint, sample, shuffle, random


def haversine_distance(lon1: float, lat1: float, lon2: float, lat2: float):
    R = 6371.0

    phi1 = radians(lon1)
    phi2 = radians(lon2)
    delta_phi = radians(lon2 - lon1)
    delta_lambda = radians(lat2 - lat1)

    a = sin(delta_phi / 2)**2 + \
        cos(phi1) * cos(phi2) * \
        sin(delta_lambda / 2)**2

    c = 2 * atan2(sqrt(a), sqrt(1 - a))

    distance = R * c
    return distance


def generate_guaranteed_pairings(city_dict: dict[str, tuple[float, float]], n: int, dist_var: tuple[float, float] = (.9, 1.1)) -> list[tuple[str, str, str]]:
    city_names = list(city_dict.keys())
    k = len(city_names)

    # Validation checks
    max_possible_pairs = (k * (k - 1)) // 2
    if n > max_possible_pairs:
        raise ValueError(f"Requested {n} pairs, but max possible unique pairs is {max_possible_pairs}.")

    if n < ceil(k / 2):
        raise ValueError(f"With {k} cities, N must be at least {ceil(k / 2)} to include every city.")

    unique_pairs = set()

    shuffled_cities = city_names.copy()
    shuffle(shuffled_cities)

    for i in range(k):
        if len(unique_pairs) < n:
            c1 = shuffled_cities[i]
            c2 = shuffled_cities[(i + 1) % k]
            pair = tuple(sorted([c1, c2]))
            unique_pairs.add(pair)

    while len(unique_pairs) < n:
        c1, c2 = sample(city_names, 2)
        pair = tuple(sorted([c1, c2]))
        unique_pairs.add(pair)

    final_pairings = []
    for c1, c2 in unique_pairs:
        dist = haversine_distance(*city_dict[c1], *city_dict[c2])
        final_pairings.append((c1, c2, round(dist * (random() * (dist_var[1] - dist_var[0]) + dist_var[0]), 2)))

    return final_pairings


def generate_additional_pairings(city_dict: dict[str, tuple[float, float]], u: int, dist_var: tuple[float, float] = (.9, 1.1)) -> list[tuple[str, str, str]]:
    city_names = list(city_dict.keys())
    additional_pairs = []

    while len(additional_pairs) < u:
        additional_pairs.append(sample(city_names, 2))

    final_additional_pairings = []
    for c1, c2 in additional_pairs:
        dist = haversine_distance(*city_dict[c1], *city_dict[c2])
        final_additional_pairings.append((c1, c2, round(dist * (random() * (dist_var[1] - dist_var[0]) + dist_var[0]), 2)))

    return final_additional_pairings

In [15]:
def gen_input(V: int, edge_p: tuple[float, float] = (.9, 1.1), stops_p: float = .05, updates_p: float = .1, dist_var: tuple[float, float] = (.9, 1.1)):
    E = round(V * (1 + edge_p[0] + random() * (edge_p[1] - edge_p[0])))
    stops = randint(2, max(2, round(V * stops_p)))
    updates = randint(3, max(3, round(V * updates_p)))
    with open(f"input{V}.txt", "w") as f:
        f.write(f"{V} {E}\n")
        citV = dict(sample(list(cities.items()), V))
        edges = [' '.join([c1, c2, str(f)])
                 for c1, c2, f in generate_guaranteed_pairings(citV, E, dist_var)]
        stops = sample(list(citV.keys()), stops)
        upd_cities = dict(it for it in citV.items() if it[0] in stops)
        upd_edges = [' '.join([c1, c2, str(f)])
                     for c1, c2, f in generate_additional_pairings(upd_cities, updates, dist_var)]
        f.write('\n'.join(edges))
        f.write(f"\nPARTENZA {choice(list(citV.keys()))}")
        f.write(f"\nDESTINAZIONE {choice(list(citV.keys()))}")
        f.write(f"\nTAPPE\n")
        f.write('\n'.join(stops))
        f.write("\nAGGIORNAMENTI\n")
        f.write('\n'.join(upd_edges))

In [16]:
for V in [10, 30, 100, 1000]:
    gen_input(V, edge_p=(1.5, 2.5), stops_p=.3, updates_p=.5, dist_var=(.5, 2))